# Week 23: AI Governance and Ethics - Fairness and Explainability

## Learning Objectives

By the end of this session, you will be able to:
1. **Audit a model for bias** using Fairlearn's `MetricFrame` and fairness metrics
2. **Mitigate bias** with Fairlearn's `ThresholdOptimizer` and understand the fairness-accuracy tradeoff
3. **Explain individual predictions** with SHAP, including the local explanation behind a single decision
4. **Connect the practice to regulation** - ECOA adverse-action notices, GDPR right to explanation, and the EU AI Act high-risk classification of credit scoring

## Prerequisites

- Completed the modeling weeks (you have trained and evaluated classifiers before)
- Watched the pre-class videos on AI ethics principles, bias in ML, fairness metrics, explainability, and the regulatory landscape (GDPR, EU AI Act)

## Session Format (~1.5 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup, Restart, Baseline Model | 12 min | Code |
| Section 1: Auditing a Credit Model for Bias | 18 min | Demo |
| Lab 1: Mitigate Bias with ThresholdOptimizer | 15 min | Lab |
| Section 2: Explaining Decisions with SHAP | 18 min | Demo |
| Lab 2: Explain a Denied Application | 15 min | Lab |
| Wrap-up and Homework | 6 min | Markdown |

## The Story So Far

All program long you have built models for Bread Financial: fraud detection,
credit scoring, customer analytics. They are accurate. But accuracy is not the
only thing that matters when a model decides whether a real person gets a loan.

Imagine your loan-approval model approves men at a much higher rate than women
with the same income and credit profile. That is not just a bad look. Under the
Equal Credit Opportunity Act (ECOA) it is illegal, and a regulator can demand a
written reason for every single denial. Today you learn the two skills that keep
a model on the right side of that line:

- **Fairness**: measure whether the model treats demographic groups equitably,
  and fix it if it does not.
- **Explainability**: produce a clear, per-applicant reason for any decision,
  which is exactly what an adverse-action notice requires.

We work on a public census dataset reframed as a Bread Financial loan-approval
model, because unlike the fraud data from prior weeks it carries the protected
attributes (such as sex) that a fairness audit needs.

## Platform

Google Colab. No GPU and no AWS credentials needed - Fairlearn and SHAP run
entirely on the CPU inside this notebook.

![Responsible AI workflow: audit, mitigate, explain](https://raw.githubusercontent.com/axel-sirota/bread-financial-academy/main/exercises/week_23_ai_governance_ethics/diagrams/responsible_ai_flow.png)

## Section 0: Setup and a Baseline Credit Model

We install a pinned set of libraries, then load the data and train a plain
baseline loan-approval model. That baseline is the thing we will audit. We pin
numpy below 2 so the fairness and explainability tooling installs cleanly.

Run the next cell first to install and import everything, then continue down
the notebook.

In [ ]:
# Install the libraries for this notebook. We pin numpy below 2 so the fairness
# and explainability tooling (Fairlearn, SHAP) installs and imports cleanly.

!pip install -q \
    "numpy<2" \
    "scikit-learn>=1.4,<1.7" \
    "fairlearn>=0.12" \
    "shap>=0.46" \
    "pandas>=2.0" \
    "matplotlib>=3.7"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Confirm we are on numpy below 2.
print("numpy version:", np.__version__)

# Load the census dataset via Fairlearn. We reframe it as a Bread Financial
# loan-approval problem: target = 1 means "approve" (the person earns >50K, a
# proxy for creditworthiness), target = 0 means "deny". The dataset carries
# protected attributes (sex, race) that a fairness audit needs.
from fairlearn.datasets import fetch_adult

data = fetch_adult(as_frame=True)
X_raw = data.data.copy()
# Target arrives as the strings ">50K" / "<=50K"; turn it into 1 = approve.
y = (data.target == ">50K").astype(int)

# The protected attribute we audit on. ECOA names sex as a protected class.
sensitive = X_raw["sex"]

# Pick a small, readable set of features for a fast, interpretable model.
numeric_features = ["age", "hours-per-week", "education-num"]
categorical_features = ["workclass", "marital-status", "occupation"]
model_features = numeric_features + categorical_features

# One preprocessing + model pipeline so the demo stays simple.
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
credit_model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)

# Train/test split, keeping the sensitive feature aligned with each split.
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X_raw[model_features], y, sensitive, test_size=0.3, random_state=42
)

credit_model.fit(X_train, y_train)
y_pred = credit_model.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("This model looks fine on accuracy. Next we check if it is FAIR.")

## Section 1: Auditing a Credit Model for Bias

Accuracy is a single number for the whole population. It hides the question a
regulator actually cares about: does the model approve some groups more often
than others?

Two ideas we use today:

- **Selection rate**: the fraction of people a group that the model approves. If
  the model approves 30 percent of men but only 12 percent of women, that gap is
  a red flag.
- **Demographic parity difference**: the gap between the highest and lowest
  selection rate across groups. A value near 0 means groups are approved at
  similar rates; a large value means disparate impact.

Fairlearn gives us `MetricFrame`, which computes any metric SEPARATELY for each
group, and helper functions like `demographic_parity_difference` that summarize
the gap in one number.

```python
from fairlearn.metrics import MetricFrame, selection_rate
from fairlearn.metrics import demographic_parity_difference
```

![Fairlearn MetricFrame: per-group metrics and the parity gap](https://raw.githubusercontent.com/axel-sirota/bread-financial-academy/main/exercises/week_23_ai_governance_ethics/diagrams/fairness_metricframe.png)

In [ ]:
# DEMO: measure fairness, not just accuracy.
from fairlearn.metrics import MetricFrame, selection_rate
from fairlearn.metrics import demographic_parity_difference

# MetricFrame computes each metric separately for each value of the sensitive
# feature. Here: accuracy and selection rate, split by sex.
frame = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sens_test,
)

print("Per-group metrics (baseline model):")
print(frame.by_group)

# The selection rate gap, summarized to one number. 0 = perfect parity.
dpd = demographic_parity_difference(
    y_test, y_pred, sensitive_features=sens_test
)
print("\nDemographic parity difference (baseline):", round(dpd, 3))
print("A large gap means the model approves one group far more than the other.")

# A quick bar chart of selection rate per group makes the disparity visible.
frame.by_group["selection_rate"].plot.bar(
    title="Approval (selection) rate by sex - baseline model",
    ylabel="selection rate",
    rot=0,
)
plt.tight_layout()
plt.show()

### Think About It

The baseline model approves one group at a noticeably higher rate than the
other, even though its overall accuracy looked fine.

- If you only ever reported overall accuracy to leadership, would anyone have
  caught this? What does that say about the metrics we put on a model card?
- Demographic parity asks for equal approval rates. But what if one group is
  genuinely more creditworthy on average in the data? Is forcing equal approval
  always the right call, or can it create its own problems?

No need to write anything down. Hold these questions as we mitigate the gap next.

## Lab 1: Mitigate Bias with ThresholdOptimizer (about 15 minutes)

You measured the disparity. Now reduce it. Fairlearn's `ThresholdOptimizer`
wraps your already-trained model and picks group-specific decision thresholds so
that a fairness constraint is satisfied, without retraining the underlying model.

**Your tasks:**

1. Create a `ThresholdOptimizer` that wraps the trained `credit_model`, using the
   demographic parity constraint. Because the model is already fitted, set the
   flag that tells the optimizer not to refit it.
2. Fit the optimizer on the TRAINING data. Remember that fairness mitigation
   needs the sensitive feature passed in explicitly.
3. Produce mitigated predictions on the TEST data. The optimizer also needs the
   sensitive feature at prediction time (a point worth noticing - the protected
   attribute must be available when the decision is made).
4. Recompute the demographic parity difference on the mitigated predictions and
   compare it to the baseline value.

**Expected outcome:** the demographic parity difference should drop noticeably
compared to the baseline you computed in the demo.

**Hints:**
- The constructor takes the estimator, a `constraints` string, and a flag for an
  already-fitted estimator.
- Both `fit` and `predict` take `sensitive_features` as a keyword argument.
- Reuse `demographic_parity_difference` from the demo.

### Stretch goal (fast finishers)

Re-run the mitigation with the `equalized_odds` constraint instead of demographic
parity, and compare the two. They optimize for different definitions of fairness:
demographic parity equalizes approval RATES, while equalized odds equalizes error
rates (true positive and false positive rates) across groups. Which one changed
the predictions more? Use `equalized_odds_difference` to measure it.

### Homework Extension

Measure the PRICE of fairness. Compute the accuracy of the mitigated model and
compare it to the baseline accuracy. Mitigation usually costs a little accuracy -
quantify how much here, and write two sentences on whether that tradeoff is
acceptable for a lending decision and who at Bread Financial should make that
call.

In [ ]:
# Lab 1 SOLUTION: mitigate the measured bias with ThresholdOptimizer.
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference

# Task 1: wrap the already-fitted credit_model. constraints chooses the fairness
# definition; prefit=True tells the optimizer not to retrain the base model.
mitigator = ThresholdOptimizer(
    estimator=credit_model,
    constraints="demographic_parity",
    prefit=True,
)

# Task 2: fit the optimizer. It needs the sensitive feature to learn one
# decision threshold per group.
mitigator.fit(X_train, y_train, sensitive_features=sens_train)

# Task 3: predict on the test set. The optimizer applies the group-specific
# threshold, so it needs the sensitive feature at prediction time too.
mitigated_pred = mitigator.predict(X_test, sensitive_features=sens_test)

# Task 4: recompute the parity gap on the mitigated predictions.
mitigated_dpd = demographic_parity_difference(
    y_test, mitigated_pred, sensitive_features=sens_test
)

print("Mitigated demographic parity difference:", round(mitigated_dpd, 3))
print("This should be much closer to 0 than the baseline gap.")

## Section 2: Explaining Decisions with SHAP

A fair model still has to explain itself. When Bread Financial denies a loan,
ECOA requires a written adverse-action notice listing the main reasons for the
denial. "The model said no" is not a legal reason. We need the specific factors
that drove THIS applicant's decision.

SHAP (SHapley Additive exPlanations) assigns each feature a contribution value
for a single prediction: how much did `age`, `hours-per-week`, `education-num`,
and the rest each push the decision toward approve or deny, starting from the
average applicant.

Two views:
- **Global**: across everyone, which features matter most to the model overall.
- **Local**: for one applicant, exactly which features drove their decision.
  This local view is what an adverse-action notice is built from.

Because our credit model is a linear logistic model, SHAP can compute exact
contributions quickly with `shap.LinearExplainer`.

```python
import shap
explainer = shap.LinearExplainer(model, background_data)
```

![SHAP: global and local explanations for a decision](https://raw.githubusercontent.com/axel-sirota/bread-financial-academy/main/exercises/week_23_ai_governance_ethics/diagrams/shap_flow.png)

In [ ]:
# DEMO: explain the model globally with SHAP.
import shap

# SHAP works on numeric arrays, so we transform the data through the fitted
# preprocessing step and explain the LogisticRegression at the end of the
# pipeline. The transformed training data is the "background" SHAP compares to.
prep = credit_model.named_steps["prep"]
clf = credit_model.named_steps["clf"]

X_train_enc = prep.transform(X_train)
X_test_enc = prep.transform(X_test)

# LinearExplainer gives exact SHAP values for a linear model, fast.
# .toarray() handles the sparse output of the one-hot encoder.
explainer = shap.LinearExplainer(clf, X_train_enc.toarray())
shap_values = explainer(X_test_enc.toarray())

# Readable feature names from the preprocessing step (numeric + one-hot columns).
feature_names = prep.get_feature_names_out()
shap_values.feature_names = list(feature_names)

# Global view: which features matter most across all test applicants.
shap.plots.bar(shap_values, max_display=10)
print("These are the features that drive approvals and denials overall.")

### Think About It

Look at the global SHAP bar chart.

- Suppose one of the top drivers were a feature that correlates strongly with a
  protected attribute (for example, occupation correlating with sex). The model
  never sees sex directly, yet it could still produce a disparate outcome. How
  would your fairness audit from Section 1 catch this, when looking at the model
  internals alone would not?
- An adverse-action notice has to be understandable to the applicant, not just
  to a data scientist. How would you translate a SHAP value into a sentence a
  customer would accept as a reason?

Hold these as you build a single-applicant explanation in the next lab.

## Lab 2: Explain a Denied Application (about 15 minutes)

A specific applicant was denied. Produce the per-applicant explanation that an
adverse-action notice is built from.

**Your tasks:**

1. From the test set, find the index of one applicant the baseline model DENIED
   (predicted 0).
2. Produce a local SHAP explanation for that single applicant and display it as a
   waterfall plot. The waterfall shows each feature pushing the decision up
   (toward approve) or down (toward deny).
3. Read off the two or three features that pushed this person most toward denial.
   Those are the "main reasons" an adverse-action notice would list.

**Expected outcome:** a waterfall plot for one denied applicant, and a short
printed list of the top factors driving that denial.

**Hints:**
- You already built `shap_values` for the whole test set in the demo. Index into
  it to get a single applicant.
- `shap.plots.waterfall` takes one applicant's SHAP explanation.
- To find a denied applicant, look at where `y_pred` equals 0.

### Stretch goal (fast finishers)

Pick one APPROVED applicant and one DENIED applicant and put their waterfall
plots side by side. Which features flipped the outcome? A contrastive
explanation ("approved because X, denied because Y") is often clearer to a
customer than a single plot.

### Homework Extension

Write the actual adverse-action notice. Using the top denial factors from your
waterfall plot, draft a 3 to 4 sentence plain-English letter to the applicant
that states the main reasons for the decision, the way ECOA requires. Avoid
jargon: a customer should understand it without knowing what a SHAP value is.

In [ ]:
# Lab 2 SOLUTION: per-applicant explanation for an adverse-action notice.
import numpy as np

# Task 1: first test-set position where the baseline model predicted deny (0).
denied_idx = int(np.where(y_pred == 0)[0][0])

# Task 2: waterfall plot for that one applicant. Indexing shap_values[i] selects
# a single explanation object that the waterfall plot understands.
shap.plots.waterfall(shap_values[denied_idx], max_display=10)

# Task 3: rank this applicant's features by how strongly they pushed toward
# denial. Negative SHAP values push the prediction down (toward deny), so we sort
# ascending and take the most negative contributions.
contributions = shap_values[denied_idx].values
names = shap_values.feature_names
order = np.argsort(contributions)  # most negative (toward deny) first
top_factors = [names[j] for j in order[:3]]

print("Applicant index:", denied_idx)
print("Top denial factors (the 'main reasons' for an adverse-action notice):")
for f in top_factors:
    print(" -", f)

## Wrap-up: Responsible AI in One Workflow

Today you wrapped a responsible-AI layer around a model like the ones you have
built all program:

1. **Audit** - `MetricFrame` and `demographic_parity_difference` revealed that an
   accurate model can still approve one group far more than another.
2. **Mitigate** - `ThresholdOptimizer` shrank that gap, and your homework
   measures what it cost in accuracy (the fairness-accuracy tradeoff is a
   business decision, not a purely technical one).
3. **Explain** - SHAP turned a single denial into a specific, per-applicant list
   of reasons, which is exactly what an ECOA adverse-action notice requires.

### Where this meets regulation

- **ECOA (US lending)**: every credit denial needs the specific principal reasons.
  Your SHAP local explanation is the engine behind that notice.
- **GDPR (EU)**: the right to explanation for automated decisions. Same tooling.
- **EU AI Act**: credit scoring is classified as a high-risk use of AI, which
  brings documentation, fairness, and transparency obligations. The audit and
  explanation you ran today are the kind of evidence those obligations expect.

### Homework

1. **Fairness-accuracy tradeoff (from Lab 1)**: compute the mitigated model's
   accuracy, compare to baseline, and write two sentences on whether the tradeoff
   is acceptable for lending and who should sign off on it.
2. **Adverse-action notice (from Lab 2)**: draft the plain-English denial letter
   from your top SHAP factors.
3. **Optional deep-dive**: open `week_23_optional_aif360_lime.ipynb` to compare
   IBM's AI Fairness 360 metrics with Fairlearn, and try LIME as a second
   explainability method alongside SHAP.

### Looking Ahead to Week 24 (Capstone)

In the capstone you bring together everything from the program. Add a
responsible-AI checklist to your capstone model: run a fairness audit, mitigate
if you find disparate impact, and be ready to explain any single prediction to a
stakeholder. A model that cannot be audited or explained is not finished.